In [ ]:
import requests
import time




In [ ]:
GITHUB_HEADERS = {
    "Accept":     "application/vnd.github+json",
    "User-Agent": "ZHAW-BigData-Explorer/1.0",
}

last_etag: str  = ""

GITHUB_API_URL = "https://api.github.com/events"
etag_headers = {"If-None-Match": last_etag} if last_etag else {}

page = 0

resp = requests.get(
    GITHUB_API_URL,
    headers={**GITHUB_HEADERS, **etag_headers},
    params={"per_page": 30, "page": page},
    timeout=10,
)


In [ ]:
resp


In [ ]:
print(resp.status_code)
print(resp.headers)
print(resp.text)
print(resp.json())


In [ ]:
# Basis-Setup: Imports und Anzeigeoptionen
import os
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


# 1) Build connection (Docker-Compose compatible)
# Priority: full URL from ENV -> otherwise build from individual values

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")        # docker-compose maps 5432:5432
DB_NAME = os.getenv("DB_NAME", "github_events")
DB_USER = os.getenv("DB_USER", "github")
DB_PASSWORD = os.getenv("DB_PASSWORD", "github_secret")

DATABASE_URL = os.getenv("DATABASE_URL") or (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"DB connection ready: {DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
source_table = "events"
query = f"SELECT * FROM {source_table}"
with engine.connect() as conn:
    repo_df = pd.read_sql(text(query), conn)

print(f"Quelle: {source_table} | Geladene Zeilen: {len(repo_df):,}")
repo_df